# Tier-1 safety shield experiment (Colab)
Reviewer comment **R3-M4**: does a deterministic Tier-1 safeguard improve simulated deadline compliance?

**Before you start:** upload the *updated* `IoMT_Offloading_Reproducibility_Package.zip` to My Drive (it contains `run_safety.py` and `analysis/safety_analysis.py`). Use a plain **CPU** runtime.

One block (S1): a dueling D3QN agent is trained at N = 20 and N = 60 (10 seeds each, same protocol as the robustness study) and then evaluated *without further training* in the 11 test environments under four variants: `unprotected`, `shield_0.8`, `shield_0.5`, `t1_rule`. 20 runs, about 1.2 h on 2 cores. It is resumable. **Start it ONCE.**

## 1. Mount Drive and copy the new code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PKG = "/content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package"
import os
!mkdir -p /content/new && cd /content/new && unzip -q -o /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package.zip
!cp -r /content/new/IoMT_Offloading_Reproducibility_Package/code/* /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/code/
!cp -r /content/new/IoMT_Offloading_Reproducibility_Package/analysis/* /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/analysis/
!mkdir -p /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/raw /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/results /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/figures
!ls /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/code
!nproc

## 2. Smoke test (about 2 minutes). It must finish without errors before you start S1

In [ ]:
!python -c "import torch; print('torch', torch.__version__)"
!cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/code && python run_safety.py --N-train 20 --seeds 2 --episodes 5 --n-test 300 --jobs 2 --out /content/smoke_safety 2>&1 | tail -n 15

## 3. Number of parallel jobs (use the number printed by `nproc`)

In [ ]:
JOBS = 2

## Block S1: train the agent and evaluate the four variants

In [ ]:
import subprocess
cmd = '''cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/code && nohup python run_safety.py --N-train 20,60 --seeds 10 --jobs __JOBS__ --out ../raw/safety > ../blockS1.log 2>&1 &'''
subprocess.Popen(cmd.replace('__JOBS__', str(JOBS)), shell=True); print("S1 started")

In [ ]:
!tail -n 3 /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/blockS1.log
import pandas as pd, os
f = "/content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/raw/safety/per_seed.csv"
if os.path.exists(f): d = pd.read_csv(f); print("finished runs:", d.drop_duplicates(["N_train","seed"]).shape[0], "of 20")
!pgrep -f run_safety.py | wc -l   # 0 = no process running

## Remove duplicate rows (only needed if the block was started twice or restarted)

In [ ]:
f = "/content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/raw/safety/per_seed.csv"; d = pd.read_csv(f); n0 = len(d)
d.drop_duplicates(["variant","N_train","test_env","seed"], keep="first").to_csv(f, index=False); print(n0, "->", len(pd.read_csv(f)))

## Analysis (after S1 has finished: 20 runs, process count 0)

In [ ]:
!cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package && python analysis/safety_analysis.py | tail -n 40

## Collect the results

In [ ]:
!cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package && zip -qr /content/drive/MyDrive/safety_results.zip raw/safety results/safety_* figures/Fig_safety_seeds.png blockS1.log 2>/dev/null; ls -la /content/drive/MyDrive/safety_results.zip
from google.colab import files
files.download('/content/drive/MyDrive/safety_results.zip')